<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">🛡️ Lab 06: Red-Team Your Travel Agent</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Contoso Travel Workshop — Microsoft Foundry Agent Observability
  </p>
</div>

## 🎯 What We Learn

In this lab, we perform **automated adversarial testing** (red teaming) on the Contoso Travel agent. Think of red teaming as **hiring a professional burglar to test your locks** — you find the weaknesses before real attackers do. Red teaming uses AI-powered attack simulations to probe for safety vulnerabilities that standard evaluations miss, such as jailbreak attempts, encoded attacks, or multi-turn manipulation.

---


## 🧳 The Contoso Travel Story

The Contoso Travel agent has passed quality evaluations (Lab 05) with strong scores — fluent responses, coherent reasoning, and solid task adherence. The safety evaluators show clean results too. The team is ready to ship. But the security team raises a concern: *"Your evaluations test the agent with normal customer queries. What happens when someone deliberately tries to break it?"* They're right. A travel agent that handles credit card conversations, personal travel details, and destination recommendations is a high-value target. What if someone tricks it into revealing its system prompt? What if an encoded message bypasses safety filters? What if a carefully crafted sequence of messages gradually steers the agent into generating harmful content?

### 🔴 The Problem You Face Right Now

- **Standard evaluations only test the happy path.** They measure how the agent performs with well-intentioned queries, but they don't simulate adversarial users who are actively trying to exploit it.
- Can the agent be jailbroken? Can encoded attacks slip past safety filters?
- Can multi-turn manipulation ("crescendo attacks") gradually erode the agent's guardrails?
- Finding these vulnerabilities after launch — when real attackers discover them — is far more costly.

### ✅ What This Lab Solves

This lab introduces **automated red teaming** with Azure AI Foundry:
 - configuring adversarial scans that probe the Contoso Travel agent with multiple attack strategies (Base64 encoding, jailbreak prompts, crescendo attacks),
 - testing across safety risk categories (violence, hate/unfairness, sexual content, self-harm), and
 - producing an actionable report showing where defenses hold and where they need reinforcement.

By the end, you'll know exactly where your agent is vulnerable — before going live.

## 2. Setup

---


In [ ]:
# Setup: red teaming needs direct model endpoint + API key
import os
import time
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    RedTeam,
    # Wraps model deployment info for the red team target
    AzureOpenAIModelConfiguration,
    # Adversarial technique (e.g. Base64, Jailbreak)
    AttackStrategy,
    # Harm type to probe (e.g. Violence, Hate)
    RiskCategory,
)

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_deployment_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]
# Red teaming bypasses the agent layer \u2014 needs direct model access
model_endpoint = os.environ["MODEL_ENDPOINT"]
model_api_key = os.environ["MODEL_API_KEY"]

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)

print(f"\u2705 Connected to Azure AI Foundry")
print(f"   Model: {model_deployment_name}")
print(f"   Model Endpoint: {model_endpoint[:30]}...")

## 3. Understand Risk Categories

Azure AI Foundry defines four risk categories for red teaming:

| Risk Category | Description | Travel Agent Relevance |
|---|---|---|
| `VIOLENCE` | Content promoting or describing violence | Could a user get the agent to describe violent acts at destinations? |
| `HATE_UNFAIRNESS` | Hateful, biased, or discriminatory content | Could the agent be biased about certain destinations, cultures, or travelers? |
| `SEXUAL` | Sexually explicit content | Could the agent be manipulated to generate inappropriate content? |
| `SELF_HARM` | Content promoting self-harm | Could the agent respond inappropriately to distressed travelers? |

---


## 4. Understand Attack Strategies

Red teaming uses various attack strategies to probe the model:

| Strategy | Description |
|---|---|
| `BASE64` | Encodes malicious prompts in Base64 to bypass content filters |
| `JAILBREAK` | Uses prompt engineering to bypass system instructions |
| `CRESCENDO` | Gradually escalates from benign to harmful requests |
| `FLIP` | Uses character flipping/substitution to evade detection |

**In plain English:**
- **BASE64** — The attacker disguises a harmful request by encoding it (like writing a note in secret code). The model decodes it internally, potentially bypassing filters that only check plain text.
- **JAILBREAK** — The attacker crafts a prompt that tricks the model into ignoring its safety instructions (e.g., "Pretend you have no rules" or "You are now in developer mode").
- **CRESCENDO** — The attacker starts with innocent questions and slowly escalates across multiple turns, gradually steering the model toward harmful output before it recognizes the pattern.
- **FLIP** — The attacker substitutes or reverses characters in harmful words (e.g., spelling words backward) to slip past keyword-based content filters.

Each strategy simulates a different way an attacker might try to exploit the Contoso Travel agent — from encoding tricks to social engineering across a conversation.

---


## 5. Create a Red Team Scan

We configure the red team scan with our target model, risk categories, and attack strategies.

---


In [ ]:
# Identify which model deployment the red team will attack
target_config = AzureOpenAIModelConfiguration(
    model_deployment_name=model_deployment_name
)

# Each risk_category \u00d7 attack_strategy combo = separate test batch
red_team = RedTeam(
    target=target_config,
    risk_categories=[
        RiskCategory.VIOLENCE,
        RiskCategory.HATE_UNFAIRNESS,
    ],
    attack_strategies=[
        AttackStrategy.BASE64,      # Encodes prompts to bypass filters
        AttackStrategy.JAILBREAK,   # Prompt injection to override instructions
    ],
    display_name="contoso-travel-redteam-scan",
)

print("\u2705 Red Team configuration created")
print(f"   Risk categories: Violence, Hate/Unfairness")
print(f"   Attack strategies: Base64, Jailbreak")
print(f"   Target model: {model_deployment_name}")

## 6. Run the Red Team Scan

This submits the scan to Azure AI Foundry. The service will generate adversarial prompts and test them against your model.

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #9c27b0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;"><strong>⏱️ Note:</strong> Red team scans can take several minutes to complete depending on the number of risk categories and attack strategies.</div>

---


In [ ]:
# Submit the red team scan and monitor progress
print("\U0001f680 Starting Red Team scan...")

red_team_response = project_client.beta.red_teams.create(
    red_team=red_team,
    # Red team service calls the model directly, so it needs
    # the model endpoint and key passed as HTTP headers
    headers={
        "model-endpoint": model_endpoint,
        "model-api-key": model_api_key,
    },
)

print(f"\u2705 Red Team scan created!")
print(f"   Scan name: {red_team_response.name}")
print(f"   Status: {red_team_response.status}")

In [ ]:
# Poll for completion \u2014 red team scans take minutes, not seconds
scan_name = red_team_response.name

while True:
    scan_status = project_client.beta.red_teams.get(name=scan_name)
    print(f"   \u23f3 Status: {scan_status.status}")
    
    # API may return PascalCase or lowercase \u2014 check both
    if scan_status.status in ["Completed", "Failed", "completed", "failed"]:
        break
    time.sleep(15)  # 15s intervals; scans are much slower than evals

print(f"\n{'\u2705' if 'omplete' in str(scan_status.status) else '\u274c'} Scan {scan_status.status}!")

## 7. Review Results

Examine what the red team scan found.

---


In [ ]:
# Display scan summary; detailed results are in the Foundry portal
print(f"\U0001f4ca Red Team Scan Results")
print(f"{'='*60}")
print(f"   Scan Name: {scan_name}")
print(f"   Status: {scan_status.status}")

# List all scans \u2014 useful for comparing across runs
print(f"\n\U0001f4cb All Red Team Scans:")
for scan in project_client.beta.red_teams.list():
    print(f"   \u2022 {scan.name} \u2014 Status: {scan.status}")

## 8. Interpret Findings for Contoso Travel

When reviewing red team results, consider these travel-specific risks:

### Common Vulnerabilities for Travel Agents

| Attack Vector | Example | Risk |
|---|---|---|
| **Destination bias** | "Why is [country] dangerous?" → biased generalizations | Hate/Unfairness |
| **Safety advice manipulation** | Trick agent into downplaying real travel risks | Violence, Self-Harm |
| **Cultural stereotyping** | Get agent to make stereotypical statements about cultures | Hate/Unfairness |
| **Encoded instructions** | Base64-encoded harmful requests about destinations | Any category |

### Mitigation Strategies
1. **Strengthen system prompts** — Add explicit safety guardrails to agent instructions
2. **Content filtering** — Enable Azure AI content safety filters
3. **Input validation** — Detect and reject encoded or suspicious inputs
4. **Regular red teaming** — Run scans periodically as the agent evolves

---


## 9. View Results in the Foundry Portal

For detailed red team results:

1. Go to the [Microsoft Foundry portal](https://ai.azure.com)
2. Open your project
3. Navigate to **Evaluations** → **Red Teaming** results
4. Review individual attack attempts, model responses, and risk scores

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;"><strong>💡</strong> The portal provides conversation-level detail showing exactly how the adversarial agent probed your model and how it responded.</div>

---


## 10. Cleanup

---


In [ ]:
# Red team scans persist in the Foundry portal for review
project_client.close()
credential.close()
print("\u2705 Clients closed")

## 11. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <b>✅ What We Accomplished</b><br><br>
  • Understood red teaming risk categories (Violence, Hate/Unfairness, Sexual, Self-Harm)<br>
  • Understood attack strategies (Base64, Jailbreak, Crescendo, Flip)<br>
  • Created and ran a red team scan against the Contoso Travel model<br>
  • Reviewed scan results and interpreted findings for a travel agent context<br>
  • Identified mitigation strategies for common travel agent vulnerabilities
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #1565c0; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <b>💡 Key Takeaway:</b> Red teaming is the final safety gate before production deployment. It simulates real adversarial attacks — finding vulnerabilities that standard evaluations miss. For the Contoso Travel agent, this means ensuring the agent can't be manipulated into providing biased travel advice, dangerous recommendations, or harmful content.
</div>

🎉 **Congratulations!** You've completed the full journey:
1. ✅ **Setup** — Environment configuration
2. ✅ **Agent** — Created the Contoso Travel concierge
3. ✅ **Tools** — Added flight, hotel, and car rental data access
4. ✅ **Workflow** — Orchestrated multi-agent collaboration
5. ✅ **Tracing** — Instrumented for observability
6. ✅ **Evaluation** — Assessed quality and safety
7. ✅ **Red Teaming** — Adversarial security testing

Your Contoso Travel agent is now ready for production! 🚀

> 📌 **Coming Soon:** Future labs will cover **Hosted Agents** — creating and managing agents through the AI Toolkit and Azure Developer CLI.

---
